練習問題 8-1で用いたデータに対してラッソ回帰を評価してください。sklearn_linearモジュールのLassoクラスを使います。なお、Lassoクラスにはパラメータ設定し、変更できるので調べてみてください。具体的には以下の公式ドキュメントをみてください。
https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html#sklearn.linear_model.Lasso

In [9]:
# インポート
import requests, zipfile
import io
import pandas as pd
import numpy as np
import sklearn

In [10]:
# 自動車価格データを取得
url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data'
res = requests.get(url).content
# 取得したデータを DataFrame オブジェクトとして読み込み
auto = pd.read_csv(io.StringIO(res.decode('utf-8')), header=None)
# データの列にラベルを設定
auto.columns =['symboling','normalized-losses','make','fuel-type' ,'aspiration','num-of-doors', 
               'body-style','drive-wheels','engine-location','wheel-base','length','width','height',
               'curb-weight','engine-type','num-of-cylinders','engine-size','fuel-system','bore',
               'stroke','compression-ratio','horsepower','peak-rpm','city-mpg','highway-mpg','price']

In [11]:
# それぞれのカラムに ? が何個あるかカウント
auto = auto[['price','width','engine-size']]
auto.isin(['?']).sum()

# ? を NaN に置換して、NaN がある行を削除
auto = auto.replace('?', np.nan).dropna()
print(' 自動車データの形式 :{}'.format(auto.shape))

 自動車データの形式 :(201, 3)


In [12]:
auto = auto.assign(price=pd.to_numeric(auto.price))
print(' データ型の確認（型変換後）\n{}'.format(auto.dtypes))

 データ型の確認（型変換後）
price            int64
width          float64
engine-size      int64
dtype: object


In [16]:
# データ分割（訓練データとテストデータ）のためのインポート
from sklearn.model_selection import train_test_split
# 重回帰のモデル構築のためのインポート
from sklearn.linear_model import Lasso
# 目的変数に price を指定、説明変数にそれ以外を指定
X = auto.drop('price', axis=1)
y = auto['price']
# 訓練データとテストデータに分ける
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=0)
# 重回帰クラスの初期化と学習
model = Lasso(alpha=0.1)
model.fit(X_train,y_train)
# 決定係数を表示
print(' 決定係数 (train):{:.3f}'.format(model.score(X_train,y_train)))
print(' 決定係数 (test):{:.3f}'.format(model.score(X_test,y_test)))
# 回帰係数と切片を表示
print('\n 回帰係数 \n{}'.format(pd.Series(model.coef_, index=X.columns)))
print(' 切片 : {:.3f}'.format(model.intercept_))

 決定係数 (train):0.783
 決定係数 (test):0.778

 回帰係数 
width          1261.772805
engine-size     109.525090
dtype: float64
 切片 : -84062.884
